# Solution Evaluation Dashboard

Visualise a single `output_payload` JSON solution.

- **Gantt chart** — driver timelines (free time / driver move / service labor)
- **Distance per service** — stacked bar (labor + move)
- **Distance per driver** — stacked bar (labor + move)
- **Summary metrics** — from an optional evaluation report

All logic lives in `src/analysis/solution_evaluation.py`. Edit the **Configuration** cell only.

In [1]:
from __future__ import annotations
import json as _json
import sys
from pathlib import Path
from typing import Optional

import pandas as pd


from alfred.analysis.solution_evaluation import (
    load_payload,
    filter_by_date,
    flatten_labors,
    build_coord_lookups,
    build_points_lookup_from_rows,
    build_preassigned_lookup,
    recompute_move_distances,
    infer_missing_durations,
    reconstruct_timeline,
    compute_payload_summary,
    build_gantt_figure,
    build_service_distance_figure,
    build_driver_distance_figure,
    build_summary_table,
    build_route_map,
)
from alfred.data.parsing.driver_directory_parser import DriverDirectoryParser
from alfred.optimization.settings.model_params import ModelParams
from alfred.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

_PROJECT_ROOT = Path("../..").resolve()


In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────────────

# SOL_PAYLOAD    = _PROJECT_ROOT / "misc" / "experiments" / "scenarios" / "20260508" / "output" / "output_payload.json"
# SOL_PAYLOAD    = _PROJECT_ROOT / "misc" / "experiments" / "phase8" / "offline" / 'results' /"output" / "output_payload.json"
SOL_PAYLOAD    = _PROJECT_ROOT / "misc" / "experiments" / "phase8" / "api_snapshots" / 'pre' / "optimization_input_snapshot.json"
EVAL: Optional[Path] = None  # e.g. _PROJECT_ROOT / "data/runs/run-X/diagnostics/solution_evaluation_report.json"
LABEL          = "solution"
# PLANNING_DATE: Optional[str] = "2026-03-24"  # e.g. "2026-02-18"
PLANNING_DATE: Optional[str] = None  # e.g. "2026-02-18"

# ── Coordinate recomputation ─────────────────────────────────────────────────────────────
INPUT_FILE: Optional[Path] = None  # e.g. _PROJECT_ROOT / "data/examples/input.json"
DRIVER_DIRECTORY: Optional[Path] = _PROJECT_ROOT / "misc" / "experiments" / "phase8" / "api_snapshots" / "driver_directory_snapshot.json"
DISTANCE_METHOD: str = DEFAULT_DISTANCE_METHOD

_params = ModelParams()
ALFRED_SPEED_KMH: float = _params.alfred_speed_kmh
print(f"alfred_speed_kmh={ALFRED_SPEED_KMH} | distance_method={DISTANCE_METHOD!r}")

alfred_speed_kmh=20.0 | distance_method='osrm'


In [3]:
# ── Load and process ───────────────────────────────────────────────────────────────────
services = load_payload(SOL_PAYLOAD)
if PLANNING_DATE:
    services = filter_by_date(services, PLANNING_DATE)

# Build coordinate lookups — use INPUT_FILE when available for more accurate distances,
# fall back to service payload coordinates.
_source = load_payload(INPUT_FILE) if INPUT_FILE and INPUT_FILE.exists() else services
coord_lookup, points_lookup, vt_labor_ids = build_coord_lookups(_source, DISTANCE_METHOD)

# Build preassigned lookup from input snapshot when available.
# In the input, alfred is a raw driver id when the labor was preassigned.
# Falls back to addData.original_assigned_driver in the output payload.
_preassigned_lookup = build_preassigned_lookup(_source) if INPUT_FILE and INPUT_FILE.exists() else {}

# Pre-build driver home lookup once (used by overview and route map).
driver_home_lookup: dict = {}
if DRIVER_DIRECTORY and DRIVER_DIRECTORY.exists():
    _dir_df = DriverDirectoryParser.parse(_json.loads(DRIVER_DIRECTORY.read_text(encoding="utf-8")))
    for _, _r in _dir_df.iterrows():
        driver_home_lookup[str(_r["driver_id"])] = f"POINT({_r['longitud']} {_r['latitud']})"
if driver_home_lookup:
    print(f"[driver_home_lookup] {len(driver_home_lookup)} drivers loaded")

rows, _warnings = flatten_labors(
    services,
    coord_lookup=coord_lookup,
    vt_labor_ids=vt_labor_ids,
    preassigned_lookup=_preassigned_lookup or None,
)

# Prefer addData map coordinates (solver-exact) over service-address coordinates.
# This is the only source of geographic data when no INPUT_FILE is set.
_adddata_points = build_points_lookup_from_rows(rows)
if _adddata_points:
    points_lookup = {**points_lookup, **_adddata_points}

# Recompute move distances from merged coordinate data.
# Skipped silently when OSRM is not available and no fallback coordinates exist.
if points_lookup:
    try:
        _move_dist = recompute_move_distances(
            rows, points_lookup, DISTANCE_METHOD, driver_home_lookup or None
        )
        for r in rows:
            if r["labor_id"] in _move_dist:
                r["driver_move_distance_km"] = _move_dist[r["labor_id"]]
    except Exception as _e:
        print(f"[recompute_move_distances] skipped: {_e}")

# Infer duration_min for labors where addData was absent (e.g. direct API snapshots).
infer_missing_durations(
    rows,
    vt_speed_kmh=_params.vehicle_transport_speed_kmh,
    tiempo_alistar_min=_params.tiempo_alistar_min,
    tiempo_finalizacion_min=_params.tiempo_finalizacion_min,
)

segments = reconstruct_timeline(
    rows, ALFRED_SPEED_KMH, model_params=_params, dist_method=DISTANCE_METHOD,
    points_lookup=points_lookup, driver_home_lookup=driver_home_lookup or None,
)

drivers      = sorted({r["driver_id"]      for r in rows if r["driver_id"]  is not None})
all_services = sorted({str(r["service_id"]) for r in rows if r["service_id"] is not None})

print(f"{LABEL}: {len(rows)} labors | {len(drivers)} drivers | {len(segments)} segments")

solution: 34 labors | 14 drivers | 60 segments


---
## Solution Overview

In [4]:
_summary = compute_payload_summary(rows, tiempo_gracia_min=_params.tiempo_gracia_min, segments=segments)
_summary["labors_non_vt"] = _summary["labors_count"] - _summary["labors_vt_count"]

_overview_keys = [
    ("services_count",                "services"),
    ("labors_count",                  "labors_total"),
    ("labors_vt_count",               "labors_vt"),
    ("labors_non_vt",                 "labors_non_vt"),
    ("drivers_count",                 "drivers_used"),
    ("labors_assigned",               "labors_assigned"),
    ("labors_preassigned",            "labors_preassigned"),
    ("labors_infeasible",             "labors_infeasible"),
    ("labors_infeasible_pct",         "labors_infeasible_pct"),
    ("labors_in_grace",               "labors_in_grace"),
    ("total_grace_min",               "total_grace_min"),
    ("labors_reassignment_candidate", "reassignment_candidates"),
    ("total_labor_distance_km",       "total_labor_distance_km"),
    ("total_driver_move_distance_km", "total_driver_move_distance_km"),
    ("total_distance_km",             "total_distance_km"),
    ("avg_labor_distance_km",         "avg_labor_distance_km"),
    ("avg_driver_move_distance_km",   "avg_driver_move_distance_km"),
]
pd.DataFrame(
    [{"metric": label, "value": _summary.get(key)} for key, label in _overview_keys]
).set_index("metric")

,value
metric,
services,30.00
labors_total,34.00
labors_vt,32.00
labors_non_vt,2.00
drivers_used,14.00
labors_assigned,30.00
labors_preassigned,0.00
labors_infeasible,0.00
labors_infeasible_pct,0.00


---
## Gantt Chart — Driver Timelines

In [5]:
build_gantt_figure(segments, drivers, LABEL).show()

---
## Distance per Service

In [25]:
build_service_distance_figure(rows, all_services, LABEL).show()

---
## Distance per Driver

In [26]:
build_driver_distance_figure(rows, drivers, LABEL).show()

---
## Route Map

Select a driver from the dropdown to visualise their geographic route.

Segment colours:
- **Blue** — service labor (vehicle transportation)
- **Orange** — driver move (inter-service travel)
- Dashed lines indicate straight-line fallback when OSRM is unavailable.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

_drv_dropdown = widgets.Dropdown(
    options=drivers,
    description="Driver:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)
_map_output = widgets.Output()


def _on_driver_change(change):
    _map_output.clear_output(wait=True)
    with _map_output:
        try:
            m = build_route_map(
                services=services,
                rows=rows,
                driver_id=change["new"],
                driver_home_wkt=driver_home_lookup.get(str(change["new"])),
                label=LABEL,
                points_lookup=points_lookup,
            )
            display(m)
        except Exception as e:
            print(f"Could not render route for driver {change['new']!r}: {e}")


_drv_dropdown.observe(_on_driver_change, names="value")
_on_driver_change({"new": _drv_dropdown.value})
display(_drv_dropdown, _map_output)

Dropdown(description='Driver:', layout=Layout(width='300px'), options=('10451', '10500', '11988', '119889', '1…

Output()

---
## Summary Metrics
_Requires `EVAL` to be set._

In [18]:
tbl = build_summary_table(EVAL, LABEL)
if tbl is not None:
    display(tbl)

No evaluation report provided — set EVAL to enable this section.
